# Potential Talents — Project Audit and Setup

Supporting executable for repository audit, controlled prototyping, cache verification and promotion. The main analysis remains in `01_Potential_Talents_Main.ipynb`.

## Block 1 — Repository discovery / inventory

In [ ]:
from pathlib import Path
import json, sys

ROOT = Path.cwd().resolve()
if not (ROOT / 'project_control').exists():
    for p in ROOT.parents:
        if (p / 'project_control').exists():
            ROOT = p
            break
print('Project root:', ROOT)
print('Python:', sys.version.split()[0])
print('Inventory:', ROOT / 'project_control/repository_inventory.csv')

## Block 2 — Current-state artifact assessment

In [ ]:
import pandas as pd

inventory = pd.read_csv(ROOT / "project_control/repository_inventory.csv")
path_col = "relative_path" if "relative_path" in inventory.columns else "path"
cols = [path_col, "lifecycle_state"]
if "authority_status" in inventory.columns:
    cols.append("authority_status")
inventory[cols]

## Block 3 — Target-state blueprint

In [ ]:
registry = json.loads((ROOT / 'project_control/project_registry.json').read_text())
assert registry['gate_state']['CONSTRUCTION_READY'] is True
print('Registry version:', registry['registry_version'])
print('Analytical target:', registry['analytical_target'])

## Block 4 — Migration / master build plan

In [ ]:
migration = pd.read_csv(ROOT / 'project_control/migration_matrix.csv')
print('Migration rows:', len(migration))
display(migration[['current_path','lifecycle_state','target_path_or_replacement','migration_status']])

## Block 5 — Environment / dependency verification

In [ ]:
import importlib.metadata as mdlib
from src.config import ProjectConfig, SEED, QUERIES, RIDGE_ALPHAS
from src.io_utils import sha256_file

cfg = ProjectConfig.from_root(ROOT)
assert SEED == 42
assert len(QUERIES) == 2
assert len(RIDGE_ALPHAS) == 15
lock_hash = sha256_file(ROOT / 'requirements-lock.txt')
print('SEED:', SEED)
print('Queries:', QUERIES)
print('Ridge alpha count:', len(RIDGE_ALPHAS))
print('Lock SHA256:', lock_hash)
for package in ['numpy','pandas','scikit-learn','scipy','matplotlib','pytest']:
    print(package, mdlib.version(package))

## Block 6 — Data / external-resource inspection

Validate immutable raw-source hashes/schema, the locked `104 → 52 → remove 2 → normalized dedup → 50 → HR34` path, the manual-label firewall, and Model-40 source identity. Manual labels remain appendix-only.

In [ ]:
from src.data_pipeline import load_raw_candidates, clean_candidates
from src.hr_rules import apply_hr_rules, ruleset_hash
from src.io_utils import sha256_file

RAW = ROOT / "data/raw/potential-talents.csv"
MANUAL = ROOT / "data/raw/manual_relevance_labels.csv"

raw = load_raw_candidates(RAW)
clean_50, cleaning_audits = clean_candidates(raw)
classified_50, hr34 = apply_hr_rules(clean_50)
manual_labels = pd.read_csv(MANUAL)

assert len(raw) == 104
assert len(cleaning_audits["title_universe"]) == 52
assert cleaning_audits["exclusions"]["representative_id"].astype(int).tolist() == [75, 103]
assert len(clean_50) == 50
assert int((~cleaning_audits["dedup"]["kept"]).sum()) == 0
assert len(hr34) == 34 and int((classified_50["H"] == 0).sum()) == 16
assert len(manual_labels) == 52
assert "manual_relevance_grade" not in clean_50.columns
assert "manual_relevance_grade" not in classified_50.columns
assert "manual_relevance_grade" not in hr34.columns

resource_audit = {
    "raw_sha256": sha256_file(RAW),
    "manual_label_sha256": sha256_file(MANUAL),
    "raw_rows": len(raw),
    "exact_title_universe": len(cleaning_audits["title_universe"]),
    "excluded_ids": cleaning_audits["exclusions"]["representative_id"].astype(int).tolist(),
    "clean_50": len(clean_50),
    "hr34": len(hr34),
    "irrelevant": int((classified_50["H"] == 0).sum()),
    "ruleset_hash": ruleset_hash(),
    "manual_label_firewall": "PASS",
}
resource_audit

## Block 7 — Helper / rule / test prototyping

Critical cleaning, rule, validation, persistence, embedding, ranking and presentation helpers are promoted only with paired tests. This block reruns the promoted foundation tests and rechecks the HR34 identity without using manual labels.

In [ ]:
import subprocess

result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests"],
    cwd=ROOT, text=True, capture_output=True
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
assert result.returncode == 0, "Promoted foundation tests failed"

EXPECTED_HR34_IDS = [1,3,4,6,7,8,10,12,13,27,28,66,67,68,69,70,71,72,73,74,76,77,78,79,81,82,83,84,88,89,94,99,100,101]
assert hr34["representative_id"].astype(int).tolist() == EXPECTED_HR34_IDS
print("HR34 identity and promoted tests validated.")

## Block 8 — Embedding / cache verification

Validate the committed compact Model-40 cache against the exact HR34/query token set, shared preprocessing identity, dimensionality, token coverage and deterministic semantic-score construction. The full external model archive is not committed.

In [ ]:
from src.config import QUERIES, LOW_NONZERO_COVERAGE_WARN_RATIO
from src.embeddings import (
    required_tokens, load_compact_cache, embedding_matrix, semantic_scores, preprocessing_hash
)

CACHE_NPZ = ROOT / "cache/model40_required_vectors.npz"
CACHE_META = ROOT / "cache/model40_cache_metadata.json"
required = required_tokens(list(hr34["job_title"].astype(str)) + list(QUERIES))
vectors, cache_meta = load_compact_cache(CACHE_NPZ, CACHE_META, expected_token_set=required)
X_embed, candidate_diag = embedding_matrix(hr34["job_title"].astype(str), vectors)
Q_embed, query_diag = embedding_matrix(QUERIES, vectors)
W_query, W = semantic_scores(X_embed, Q_embed)

assert X_embed.shape == (34, 100)
assert Q_embed.shape == (2, 100)
assert cache_meta["expected_vocab"] == 4_027_169
assert cache_meta["dimension"] == 100
assert cache_meta["preprocessing_hash"] == preprocessing_hash()
assert all(d["recognized_count"] > 0 for d in candidate_diag + query_diag)
low_nonzero = [d for d in candidate_diag if 0 < d["coverage_ratio"] < LOW_NONZERO_COVERAGE_WARN_RATIO]
assert not low_nonzero
assert all(d["coverage_ratio"] == 1.0 for d in query_diag)

H = hr34["H"].to_numpy(float)
G = (H + W) / 2.0
assert len(G) == 34 and ((G >= 0) & (G <= 1)).all()

cache_health = {
    "required_tokens": len(required),
    "cached_tokens": cache_meta["cached_token_count"],
    "oov_required_tokens": cache_meta["oov_required_tokens"],
    "candidate_min_coverage": min(d["coverage_ratio"] for d in candidate_diag),
    "query_coverages": [d["coverage_ratio"] for d in query_diag],
    "candidate_embedding_shape": list(X_embed.shape),
    "query_embedding_shape": list(Q_embed.shape),
    "W_range": [float(W.min()), float(W.max())],
    "G_range": [float(G.min()), float(G.max())],
}
cache_health

## Block 9 — Promotion / production-readiness checklist

Evaluate the current preconstruction promotion gate. At Phase 7 the data/rule/cache foundation is validated, but the authoritative main notebook remains blocked until the modeling and feedback modules/tests are promoted and the complete 102-cell construction specification is present in `project_control/`.

In [ ]:
required_now = [
    ROOT / "src/config.py", ROOT / "src/data_pipeline.py", ROOT / "src/hr_rules.py",
    ROOT / "src/embeddings.py", ROOT / "src/ranking.py", ROOT / "src/validation.py",
    ROOT / "cache/model40_required_vectors.npz", ROOT / "cache/model40_cache_metadata.json",
]
assert all(p.exists() for p in required_now)

future_required = [
    ROOT / "src/modeling.py", ROOT / "src/feedback.py",
    ROOT / "tests/test_modeling.py", ROOT / "tests/test_feedback.py",
]
missing_future = [str(p.relative_to(ROOT)) for p in future_required if not p.exists()]
MAIN_NOTEBOOK_BUILD_READY = len(missing_future) == 0
print("Phase-7 data/rule/cache foundation: VALIDATED")
print("Main-notebook build ready:", MAIN_NOTEBOOK_BUILD_READY)
print("Still required before main construction:", missing_future)